# 35 · Complemento: clima de gold (A3) para el conteo sobre la configuración final

Encargo complementario de `docs/modeling/fe_hiperparametros.md` («Encargo complementario · clima para el
conteo»). El preregistro (`src/modeling/complemento_clima.py → PREREGISTRO_COMPLEMENTO`) se commiteó antes de
esta corrida. Comparación única (la 20 del encargo): **F+A3** frente a **F∩A3**. El protocolo no cambia y nada
se escribe en `data/`.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.eda.carga import cargar_integrado
from src.modeling import cache_experimentos as cache
from src.modeling import complemento_clima as cc
from src.modeling import configuracion_final as cf
from src.modeling import diagnostico_arquitectura as d
from src.modeling import diagnostico_inicios as di
from src.modeling import experimentos_variables as ev
from src.modeling import fe_hiperparametros as fh
from src.modeling import preregistro_fe as pr
from src.modeling import protocolo_arquitectura as pa
from src.modeling import salida_probabilistica as sp
from src.modeling.seguimiento_mlflow import registrar_o_avisar
from src.utils.paths import (EXPERIMENTOS_FE_HIPERPARAMETROS, FIGURAS_FE_HIPERPARAMETROS, GOLD_MANIFEST,
                             METRICAS_FE_HIPERPARAMETROS, SILVER_INTEGRADO)

warnings.filterwarnings("ignore", message=".*mismatched devices.*")
pd.set_option("display.precision", 4)
HUELLA = cache.huella_encargo()
CACHE = EXPERIMENTOS_FE_HIPERPARAMETROS / "cache"
DESTINO = EXPERIMENTOS_FE_HIPERPARAMETROS / "complemento_clima"
DESTINO.mkdir(parents=True, exist_ok=True)
CONF = cf.CONFIGURACION_2026_V1  # F: la configuración congelada antes del complemento

panel = cargar_integrado(SILVER_INTEGRADO)
regla = json.loads(GOLD_MANIFEST.read_text(encoding="utf-8"))["regla_brote"]
ins = fh.insumos(panel, regla)
matrices, serie = ins["matrices"], ins["serie"]
origenes = cf.origenes(panel, ins, regla, configuracion=CONF)
COL_F = list(CONF["columnas"])
COL_FA3 = [*COL_F, *cc.COLUMNAS_A3]
o_a3, excluidas = ev.origenes_con(matrices, cc.COLUMNAS_A3, origenes=origenes)
assert not o_a3[COL_FA3].isna().any().any()
print("F:", len(COL_F), "columnas | F+A3:", len(COL_FA3), "| filas excluidas por A3:", excluidas,
      "| comparaciones del encargo:", pr.comparaciones()["total_encargo"])

F: 9 columnas | F+A3: 11 | filas excluidas por A3: {'filas': 65, 'primera': '2017-02-05', 'ultima': '2017-02-05'} | comparaciones del encargo: 20


In [2]:
pd.Series({k: json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
           for k, v in cc.PREREGISTRO_COMPLEMENTO.items()}, name="preregistro").to_frame()

,preregistro
fecha,"2026-10-27, antes de cualquier corrida del com..."
comparacion,20 del encargo de FE (única del complemento)
referencia,"F∩A3: configuración final (CONFIGURACION_2026,..."
candidata,F+A3: C3 + hum_rel_media_media5_h2 y precip_to...
lectura,"F completa (9 columnas, todas las filas) solo ..."
semillas,"[17, 29, 41, 53, 67]"
umbral_empate,"por métrica, horizonte y temporada, el mayor r..."
delta,mediana de las 5 semillas de F+A3 menos median...
desigualdades,estrictas
condicion_1_conteo,"{""mejora_en_h"": ""razón MAE y razón RMSE bajan ..."


## Entrenamientos (5 semillas): F+A3, F∩A3 y F completa (lectura)

In [3]:
def _evaluar(columnas, o):
    return ev.evaluar_y_resumir(matrices, serie, columnas=columnas, origenes=o, formulacion=CONF["formulacion"],
                                dispersion=CONF["dispersion"], hiperparametros=CONF["hiperparametros"])

fa3 = cache.cargar_o_calcular("complemento_F_A3", lambda: _evaluar(COL_FA3, o_a3), huella=HUELLA, directorio=CACHE)
f_int = cache.cargar_o_calcular("complemento_F_interseccion_A3", lambda: _evaluar(COL_F, o_a3), huella=HUELLA, directorio=CACHE)
f_completa = cache.cargar_o_calcular("complemento_F_completa", lambda: _evaluar(COL_F, origenes), huella=HUELLA, directorio=CACHE)

caché: 'complemento_F_A3' calculada (huella c29942a28b6f)


caché: 'complemento_F_interseccion_A3' calculada (huella c29942a28b6f)


caché: 'complemento_F_completa' calculada (huella c29942a28b6f)


## Condición 1 · conteo

In [4]:
conteo = cc.tabla_conteo(fa3["evaluaciones"], f_int["evaluaciones"])
cond_conteo = cc.condicion_conteo(conteo)
display(conteo.round(4))
cond_conteo

,horizonte,bloque,metrica,mediana_candidata,mediana_referencia,delta,umbral,baja,sube
0,2,temporada_2022,razon_mae,0.9187,1.0004,-0.0816,0.0157,True,False
1,2,temporada_2022,razon_rmse,0.9377,1.0995,-0.1618,0.0319,True,False
2,2,temporada_2023,razon_mae,0.8181,0.8843,-0.0662,0.0107,True,False
3,2,temporada_2023,razon_rmse,0.8685,0.9913,-0.1228,0.0197,True,False
4,3,temporada_2022,razon_mae,0.8831,1.0170,-0.1338,0.0287,True,False
5,3,temporada_2022,razon_rmse,0.9215,1.2213,-0.2998,0.0514,True,False
6,3,temporada_2023,razon_mae,0.7470,0.8112,-0.0642,0.0055,True,False
7,3,temporada_2023,razon_rmse,0.8180,0.9123,-0.0943,0.0108,True,False
8,4,temporada_2022,razon_mae,0.8383,1.0360,-0.1976,0.0264,True,False
9,4,temporada_2022,razon_rmse,0.9052,1.3688,-0.4636,0.0411,True,False


,horizonte,mejora_conteo,empeora_conteo
0,2,True,False
1,3,True,False
2,4,True,False


## Condición 2 · la alerta no pierde (regla simétrica del protocolo)

In [5]:
veredicto = ev.comparar_familia("F+A3", fa3["evaluaciones"], fa3["resumen"], "F∩A3", f_int["evaluaciones"], f_int["resumen"])
COLS_VEREDICTO = ["horizonte", "veredicto", "mejora_f1_050", "empeora_f1_050", "mejora_antelacion", "empeora_antelacion",
                  "vetos", "vetos_contra_referencia", "antelacion_diferencia", "antelacion_ic95_inferior",
                  "antelacion_ic95_superior", "mejora_recall_nuevos", "posible_senal_de_temporada"]
display(veredicto[COLS_VEREDICTO])
veredicto[["horizonte", *[c for c in veredicto if c.startswith(("delta_", "umbral_"))]]].round(4)

,horizonte,veredicto,mejora_f1_050,empeora_f1_050,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,antelacion_diferencia,antelacion_ic95_inferior,antelacion_ic95_superior,mejora_recall_nuevos,posible_senal_de_temporada
0,2,empate,False,False,False,False,None,razon_mae,0.0211,-0.0211,0.0632,False,False
1,3,empate,False,False,False,False,None,razon_mae,-0.0526,-0.1158,0.0000,False,False
2,4,empate,False,False,False,False,None,razon_mae,-0.0316,-0.0739,0.0105,False,False


,horizonte,umbral_f1_050_temporada_2022,umbral_f1_050_temporada_2023,umbral_auprc_temporada_2022,umbral_auprc_temporada_2023,umbral_brier_temporada_2022,umbral_brier_temporada_2023,umbral_razon_mae_temporada_2022,umbral_razon_mae_temporada_2023,delta_f1_050_temporada_2022,delta_f1_050_temporada_2023,delta_auprc_temporada_2022,delta_auprc_temporada_2023,delta_brier_temporada_2022,delta_brier_temporada_2023,delta_razon_mae_temporada_2022,delta_razon_mae_temporada_2023
0,2,0.0107,0.0019,0.0041,0.0007,0.0004,0.0006,0.0157,0.0107,0.0070,0.0058,-0.0060,0.0007,0.0010,0.0000,-0.0816,-0.0662
1,3,0.0354,0.0038,0.0040,0.0012,0.0008,0.0009,0.0287,0.0055,-0.0476,0.0017,-0.0076,0.0010,0.0020,0.0003,-0.1338,-0.0642
2,4,0.0605,0.0043,0.0054,0.0011,0.0011,0.0009,0.0264,0.0110,-0.1918,0.0092,-0.0090,0.0026,0.0053,-0.0001,-0.1976,-0.0571


## Decisión

In [6]:
decision = cc.decidir(cond_conteo, veredicto)
decision

{'condicion_1_conteo': True,
 'condicion_2_alerta_no_pierde': True,
 'horizontes_mejora_conteo': [2, 3, 4],
 'horizontes_empeora_conteo': [],
 'horizontes_pierde_alerta': [],
 'adoptada': 'F+A3'}

## Lectura (no decide)

### Métricas por horizonte y temporada (mediana y rango entre semillas)

In [7]:
lectura = pd.concat([cc.tabla_lectura(fa3["evaluaciones"], "F+A3"), cc.tabla_lectura(f_int["evaluaciones"], "F∩A3"),
                     cc.tabla_lectura(f_completa["evaluaciones"], "F")], ignore_index=True)
lectura.pivot_table(index=["bloque", "horizonte"], columns="variante",
                    values=["precision_050", "recall_050", "f1_050", "auprc", "brier", "razon_mae", "razon_rmse",
                            "mae", "rmse"]).round(3)

auprc                brier               f1_050  \
variante                      F   F+A3   F∩A3      F   F+A3   F∩A3      F   
bloque         horizonte                                                    
temporada_2022 2          0.789  0.782  0.788  0.053  0.054  0.053  0.711   
               3          0.781  0.773  0.780  0.057  0.059  0.057  0.658   
               4          0.753  0.741  0.750  0.062  0.068  0.063  0.546   
temporada_2023 2          0.947  0.948  0.947  0.062  0.063  0.063  0.852   
               3          0.928  0.930  0.929  0.075  0.075  0.075  0.828   
               4          0.917  0.920  0.918  0.084  0.084  0.084  0.803   
temporada_2024 2          0.891  0.894  0.892  0.076  0.075  0.076  0.796   
               3          0.870  0.876  0.870  0.086  0.083  0.086  0.778   
               4          0.831  0.846  0.830  0.098  0.094  0.099  0.745   

                                           mae  ... razon_mae razon_rmse  \
variante                   F+A3   F∩A3       F  ...      F∩A3          F   
bloque         horizonte                        ...                        
temporada_2022 2          0.709  0.702   1.961  ...     1.000      1.106   
               3          0.594  0.642   2.437  ...     1.017      1.232   
               4          0.333  0.525   2.958  ...     1.036      1.380   
temporada_2023 2          0.858  0.852   9.544  ...     0.884      0.987   
               3          0.828  0.826  11.842  ...     0.811      0.913   
               4          0.811  0.802  13.512  ...     0.765      0.852   
temporada_2024 2          0.803  0.796   6.699  ...     1.343      1.673   
               3          0.781  0.778   9.660  ...     1.508      1.890   
               4          0.749  0.744  12.725  ...     1.617      2.014   

                                       recall_050                  rmse  \
variante                   F+A3   F∩A3          F   F+A3   F∩A3       F   
bloque         horizonte                                                  
temporada_2022 2          0.938  1.099      0.696  0.687  0.666   6.678   
               3          0.922  1.221      0.559  0.462  0.531   9.028   
               4          0.905  1.369      0.399  0.202  0.376  11.782   
temporada_2023 2          0.869  0.991      0.812  0.821  0.811  48.836   
               3          0.818  0.912      0.774  0.768  0.769  61.201   
               4          0.779  0.851      0.724  0.733  0.721  68.286   
temporada_2024 2          0.945  1.691      0.866  0.873  0.863  26.361   
               3          0.981  1.874      0.873  0.884  0.874  39.009   
               4          1.011  2.007      0.846  0.861  0.848  51.276   

                                          
variante                    F+A3    F∩A3  
bloque         horizonte                  
temporada_2022 2           5.660   6.636  
               3           6.755   8.952  
               4           7.727  11.684  
temporada_2023 2          42.970  49.046  
               3          54.848  61.168  
               4          62.474  68.263  
temporada_2024 2          14.898  26.650  
               3          20.256  38.678  
               4          25.745  51.081  

[9 rows x 27 columns]

In [8]:
lectura.pivot_table(index=["bloque", "horizonte"], columns="variante",
                    values=[c for c in lectura if c.startswith("rango_")]).round(4)

rango_auprc                 rango_brier          \
variante                           F    F+A3    F∩A3           F    F+A3   
bloque         horizonte                                                   
temporada_2022 2              0.0017  0.0031  0.0041      0.0005  0.0003   
               3              0.0040  0.0026  0.0040      0.0005  0.0005   
               4              0.0054  0.0037  0.0054      0.0006  0.0011   
temporada_2023 2              0.0010  0.0007  0.0006      0.0006  0.0006   
               3              0.0012  0.0012  0.0008      0.0006  0.0009   
               4              0.0016  0.0011  0.0007      0.0006  0.0009   
temporada_2024 2              0.0010  0.0011  0.0009      0.0007  0.0003   
               3              0.0011  0.0011  0.0018      0.0006  0.0005   
               4              0.0014  0.0015  0.0026      0.0006  0.0010   

                                 rango_f1_050                 rango_mae  ...  \
variante                    F∩A3            F    F+A3    F∩A3         F  ...   
bloque         horizonte                                                 ...   
temporada_2022 2          0.0004       0.0074  0.0066  0.0107    0.0080  ...   
               3          0.0008       0.0183  0.0226  0.0354    0.0450  ...   
               4          0.0011       0.0084  0.0605  0.0433    0.0795  ...   
temporada_2023 2          0.0004       0.0012  0.0019  0.0013    0.1277  ...   
               3          0.0006       0.0029  0.0038  0.0030    0.1318  ...   
               4          0.0006       0.0053  0.0025  0.0043    0.2067  ...   
temporada_2024 2          0.0004       0.0034  0.0017  0.0030    0.1858  ...   
               3          0.0004       0.0019  0.0015  0.0031    0.3149  ...   
               4          0.0007       0.0015  0.0039  0.0033    0.4046  ...   

                         rango_razon_mae rango_razon_rmse                  \
variante                            F∩A3                F    F+A3    F∩A3   
bloque         horizonte                                                    
temporada_2022 2                  0.0157           0.0062  0.0157  0.0319   
               3                  0.0287           0.0329  0.0221  0.0514   
               4                  0.0264           0.0533  0.0340  0.0411   
temporada_2023 2                  0.0107           0.0232  0.0197  0.0125   
               3                  0.0055           0.0141  0.0108  0.0017   
               4                  0.0110           0.0108  0.0125  0.0112   
temporada_2024 2                  0.1020           0.1259  0.0364  0.2185   
               3                  0.1031           0.1492  0.0394  0.2165   
               4                  0.1083           0.1359  0.0529  0.2163   

                         rango_recall_050                 rango_rmse          \
variante                                F    F+A3    F∩A3          F    F+A3   
bloque         horizonte                                                       
temporada_2022 2                   0.0209  0.0348  0.0371     0.0376  0.0945   
               3                   0.0255  0.0325  0.0580     0.2413  0.1622   
               4                   0.0116  0.0441  0.0464     0.4546  0.2905   
temporada_2023 2                   0.0029  0.0019  0.0029     1.1498  0.9768   
               3                   0.0077  0.0067  0.0077     0.9463  0.7228   
               4                   0.0087  0.0067  0.0096     0.8662  1.0020   
temporada_2024 2                   0.0047  0.0082  0.0070     1.9835  0.5730   
               3                   0.0058  0.0093  0.0047     3.0805  0.8126   
               4                   0.0047  0.0035  0.0058     3.4587  1.3455   

                                  
variante                    F∩A3  
bloque         horizonte          
temporada_2022 2          0.1923  
               3          0.3764  
               4          0.3512  
temporada_2023 2          0.6196  
               3    

### Antelación sobre inicios limpios (frente a F∩A3 y frente a la persistencia, semilla 17)

In [9]:
antelacion = pd.concat({"F+A3": fa3["resumen"]["antelacion"], "F∩A3": f_int["resumen"]["antelacion"],
                        "F": f_completa["resumen"]["antelacion"]}, names=["variante"]).reset_index(level=0)
display(antelacion.pivot_table(index="variante", columns=["horizonte", "bloque"], values="mediana").round(3))
persistencia = pd.concat({"F+A3": fh.diferencia_antelacion_persistencia(fa3["evaluaciones"][17]),
                          "F∩A3": fh.diferencia_antelacion_persistencia(f_int["evaluaciones"][17])},
                         names=["variante"]).reset_index(level=0)
persistencia.round(3)

horizonte                   2                                               \
bloque    seleccion_2022_2023 temporada_2022 temporada_2023 temporada_2024   
variante                                                                     
F                       0.137          0.132          0.140          0.435   
F+A3                    0.147          0.132          0.158          0.435   
F∩A3                    0.116          0.079          0.140          0.435   

horizonte                   3                                               \
bloque    seleccion_2022_2023 temporada_2022 temporada_2023 temporada_2024   
variante                                                                     
F                       0.211          0.105          0.281          0.565   
F+A3                    0.147          0.053          0.211          0.565   
F∩A3                    0.200          0.079          0.263          0.565   

horizonte                   4                                               
bloque    seleccion_2022_2023 temporada_2022 temporada_2023 temporada_2024  
variante                                                                    
F                       0.232          0.053          0.351          0.478  
F+A3                    0.189          0.000          0.316          0.522  
F∩A3                    0.221          0.053          0.333          0.478

,variante,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2022,diferencia_temporada_2023
0,F+A3,2,95,0.021,-0.042,0.074,0.026,0.018
1,F+A3,3,95,0.011,-0.053,0.074,-0.053,0.053
2,F+A3,4,95,-0.021,-0.105,0.063,-0.263,0.140
0,F∩A3,2,95,0.000,-0.063,0.063,-0.026,0.018
1,F∩A3,3,95,0.063,-0.011,0.137,-0.026,0.123
2,F∩A3,4,95,0.011,-0.074,0.095,-0.211,0.158


### Recall en las 2 primeras semanas de brotes nuevos

In [10]:
recall = pd.concat({"F+A3": fa3["resumen"]["recall_por_tipo"], "F∩A3": f_int["resumen"]["recall_por_tipo"]},
                   names=["variante"]).reset_index(level=0)
recall.query("tipo == 'brote_nuevo'").pivot_table(index="variante", columns=["horizonte", "bloque"],
                                                  values="recall_mediana").round(3)

horizonte               2                                               \
bloque    calendario_2025 temporada_2022 temporada_2023 temporada_2024   
variante                                                                 
F+A3                  0.0          0.102          0.151          0.353   
F∩A3                  0.0          0.061          0.116          0.353   

horizonte               3                                               \
bloque    calendario_2025 temporada_2022 temporada_2023 temporada_2024   
variante                                                                 
F+A3                  0.5          0.041          0.128          0.265   
F∩A3                  0.5          0.061          0.163          0.265   

horizonte               4                                               
bloque    calendario_2025 temporada_2022 temporada_2023 temporada_2024  
variante                                                                
F+A3                  0.0          0.000          0.151          0.324  
F∩A3                  0.0          0.041          0.163          0.324

### Conteo por fase de la curva regional (subida, pico, bajada, valle)

In [11]:
BLOQUES_FASE = (*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE)
fases = pd.concat({"F+A3": di.errores_por_fase(list(fa3["evaluaciones"].values()), BLOQUES_FASE),
                   "F∩A3": di.errores_por_fase(list(f_int["evaluaciones"].values()), BLOQUES_FASE)},
                  names=["variante"]).reset_index(level=0)
fases.pivot_table(index=["bloque", "fase"], columns=["horizonte", "variante"], values="razon_mae").round(3)

horizonte                  2             3             4       
variante                F+A3   F∩A3   F+A3   F∩A3   F+A3   F∩A3
bloque         fase                                            
temporada_2022 bajada  0.587  0.571  0.459  0.434  0.386  0.380
               pico    1.024  1.203  1.075  1.405  1.076  1.598
               subida  1.069  1.138  1.056  1.141  1.040  1.065
               valle   0.879  0.877  0.818  0.820  0.725  0.742
temporada_2023 bajada  0.550  0.551  0.409  0.411  0.342  0.341
               pico    0.890  1.009  0.905  1.017  0.947  1.054
               subida  0.980  1.044  0.960  1.047  0.948  0.992
               valle   1.050  1.029  0.945  0.950  0.866  0.915
temporada_2024 bajada  0.838  1.406  0.858  1.805  0.935  2.215
               pico    1.051  1.801  1.093  1.953  1.144  1.995
               subida  0.848  0.822  0.793  0.796  0.704  0.725
               valle   0.941  0.937  0.874  0.863  0.810  0.806

In [12]:
fases.pivot_table(index=["bloque", "fase"], columns=["horizonte", "variante"], values="razon_rmse").round(3)

horizonte                  2             3             4       
variante                F+A3   F∩A3   F+A3   F∩A3   F+A3   F∩A3
bloque         fase                                            
temporada_2022 bajada  0.478  0.469  0.388  0.374  0.352  0.361
               pico    1.050  1.262  1.079  1.541  1.096  1.906
               subida  1.074  1.267  1.081  1.356  1.053  1.130
               valle   0.838  0.852  0.762  0.775  0.670  0.694
temporada_2023 bajada  0.456  0.455  0.352  0.352  0.298  0.295
               pico    0.947  1.096  0.950  1.072  0.952  1.057
               subida  0.945  1.013  0.959  1.006  1.052  1.006
               valle   1.094  1.013  0.905  0.900  0.775  0.834
temporada_2024 bajada  0.859  1.563  0.880  1.892  0.995  2.387
               pico    1.033  1.967  1.064  2.096  1.083  1.999
               subida  0.806  0.754  0.787  0.778  0.720  0.729
               valle   0.819  0.819  0.738  0.743  0.657  0.651

### HU0007-4 (F, F∩A3 y F+A3)

In [13]:
def _hu(r):
    m = r["resumen"]["medianas"]
    return sp.cumplimiento_hu0007_4(m[m.bloque.isin([*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE])])

hu = pd.concat({"F": _hu(f_completa), "F∩A3": _hu(f_int), "F+A3": _hu(fa3)}, names=["variante"]).reset_index(level=0)
hu.pivot_table(index=["bloque", "horizonte"], columns="variante",
               values=["cumple_clasificacion", "cumple_regresion", "razon_mae", "razon_rmse"]).round(3)

cumple_clasificacion           cumple_regresion       \
variante                                    F F+A3 F∩A3                F F+A3   
bloque         horizonte                                                        
temporada_2022 2                          0.0  0.0  0.0              0.0  0.0   
               3                          0.0  0.0  0.0              0.0  0.0   
               4                          0.0  0.0  0.0              0.0  0.0   
temporada_2023 2                          1.0  1.0  1.0              0.0  0.0   
               3                          0.0  0.0  0.0              0.0  1.0   
               4                          0.0  0.0  0.0              0.0  1.0   
temporada_2024 2                          1.0  1.0  1.0              0.0  0.0   
               3                          1.0  1.0  1.0              0.0  0.0   
               4                          1.0  1.0  1.0              0.0  0.0   

                              razon_mae               razon_rmse                
variante                 F∩A3         F   F+A3   F∩A3          F   F+A3   F∩A3  
bloque         horizonte                                                        
temporada_2022 2          0.0     1.001  0.919  1.000      1.106  0.938  1.099  
               3          0.0     1.021  0.883  1.017      1.232  0.922  1.221  
               4          0.0     1.040  0.838  1.036      1.380  0.905  1.369  
temporada_2023 2          0.0     0.881  0.818  0.884      0.987  0.869  0.991  
               3          0.0     0.808  0.747  0.811      0.913  0.818  0.912  
               4          0.0     0.764  0.708  0.765      0.852  0.779  0.851  
temporada_2024 2          0.0     1.352  0.937  1.343      1.673  0.945  1.691  
               3          0.0     1.515  0.938  1.508      1.890  0.981  1.874  
               4          0.0     1.622  0.951  1.617      2.014  1.011  2.007

### Diagnóstico de 2022·h4: ¿discriminación o calibración? (lectura)

AUPRC (igual con o sin calibrar, porque Platt es monótono), Brier calibrado y sin calibrar, probabilidad
calibrada media frente a la prevalencia, P / R / F1 a 0,50 y semanas brote con probabilidad calibrada en
[0,40, 0,50). El Platt de 2022 se ajusta solo con 2021.

In [14]:
diag_2022_h4 = pd.concat({"F∩A3": cc.diagnostico_celda(f_int["evaluaciones"], 4, "temporada_2022"),
                          "F+A3": cc.diagnostico_celda(fa3["evaluaciones"], 4, "temporada_2022")},
                         names=["variante"]).reset_index(level=0)
display(diag_2022_h4.round(4))
diag_2022_h4.groupby("variante").median(numeric_only=True).drop(columns="semilla").round(4).T

,variante,semilla,semanas,semanas_brote,prevalencia,probabilidad_calibrada_media,probabilidad_cruda_media,auprc,brier_calibrado,brier_sin_calibrar,precision_050,recall_050,f1_050,brote_en_banda,brote_con_alerta
0,F∩A3,17,3380,431,0.1275,0.1221,0.0988,0.7465,0.0637,0.0606,0.8659,0.3596,0.5082,90,155
1,F∩A3,29,3380,431,0.1275,0.1263,0.1003,0.7519,0.0626,0.0598,0.8700,0.4037,0.5515,87,174
2,F∩A3,41,3380,431,0.1275,0.1215,0.0995,0.7500,0.0631,0.0601,0.8710,0.3759,0.5251,84,162
3,F∩A3,53,3380,431,0.1275,0.1213,0.0993,0.7496,0.0635,0.0601,0.8800,0.3573,0.5083,86,154
4,F∩A3,67,3380,431,0.1275,0.1238,0.1002,0.7499,0.0629,0.0600,0.8718,0.3944,0.5431,83,170
0,F+A3,17,3380,431,0.1275,0.1101,0.0963,0.7373,0.0680,0.0617,0.9596,0.2204,0.3585,95,95
1,F+A3,29,3380,431,0.1275,0.1091,0.0945,0.7410,0.0684,0.0616,0.9560,0.2019,0.3333,91,87
2,F+A3,41,3380,431,0.1275,0.1063,0.0953,0.7410,0.0687,0.0613,0.9655,0.1949,0.3243,92,84
3,F+A3,53,3380,431,0.1275,0.1053,0.0947,0.7410,0.0691,0.0615,0.9620,0.1763,0.2980,93,76
4,F+A3,67,3380,431,0.1275,0.1097,0.0953,0.7393,0.0680,0.0615,0.9592,0.2181,0.3554,96,94


variante,F+A3,F∩A3
semanas,3380.0000,3380.0000
semanas_brote,431.0000,431.0000
prevalencia,0.1275,0.1275
probabilidad_calibrada_media,0.1091,0.1221
probabilidad_cruda_media,0.0953,0.0995
auprc,0.7410,0.7499
brier_calibrado,0.0684,0.0631
brier_sin_calibrar,0.0615,0.0601
precision_050,0.9596,0.8710
recall_050,0.2019,0.3759


### Robustez (solo lectura): sin las 8 primeras semanas de orígenes de 2017

In [15]:
recorte = ev.recorte_robustez(o_a3, "primeras_8_semanas_2017")
o_rec = o_a3.loc[~recorte].reset_index(drop=True)
rob_fa3 = cache.cargar_o_calcular("complemento_rob_F_A3", lambda: _evaluar(COL_FA3, o_rec), huella=HUELLA, directorio=CACHE)
rob_f = cache.cargar_o_calcular("complemento_rob_F_interseccion_A3", lambda: _evaluar(COL_F, o_rec), huella=HUELLA, directorio=CACHE)
conteo_rob = cc.tabla_conteo(rob_fa3["evaluaciones"], rob_f["evaluaciones"])
signos = (pd.concat({"sin_recorte": conteo, "sin_8_semanas_2017": conteo_rob}, names=["recorte"]).reset_index(level=0)
          .assign(signo=lambda t: np.sign(t.delta).astype(int)))
display(signos.pivot_table(index=["horizonte", "bloque", "metrica"], columns="recorte", values=["delta", "signo"]).round(4))
cc.condicion_conteo(conteo_rob)

caché: 'complemento_rob_F_A3' calculada (huella c29942a28b6f)


caché: 'complemento_rob_F_interseccion_A3' calculada (huella c29942a28b6f)


delta              \
recorte                             sin_8_semanas_2017 sin_recorte   
horizonte bloque         metrica                                     
2         temporada_2022 razon_mae             -0.0586     -0.0816   
                         razon_rmse            -0.1242     -0.1618   
          temporada_2023 razon_mae             -0.0328     -0.0662   
                         razon_rmse            -0.0396     -0.1228   
3         temporada_2022 razon_mae             -0.0883     -0.1338   
                         razon_rmse            -0.1663     -0.2998   
          temporada_2023 razon_mae             -0.0277     -0.0642   
                         razon_rmse            -0.0300     -0.0943   
4         temporada_2022 razon_mae             -0.0902     -0.1976   
                         razon_rmse            -0.1541     -0.4636   
          temporada_2023 razon_mae             -0.0101     -0.0571   
                         razon_rmse            -0.0170     -0.0722   

                                                 signo              
recorte                             sin_8_semanas_2017 sin_recorte  
horizonte bloque         metrica                                    
2         temporada_2022 razon_mae                -1.0        -1.0  
                         razon_rmse               -1.0        -1.0  
          temporada_2023 razon_mae                -1.0        -1.0  
                         razon_rmse               -1.0        -1.0  
3         temporada_2022 razon_mae                -1.0        -1.0  
                         razon_rmse               -1.0        -1.0  
          temporada_2023 razon_mae                -1.0        -1.0  
                         razon_rmse               -1.0        -1.0  
4         temporada_2022 razon_mae                -1.0        -1.0  
                         razon_rmse               -1.0        -1.0  
          temporada_2023 razon_mae                -1.0        -1.0  
                         razon_rmse               -1.0        -1.0

,horizonte,mejora_conteo,empeora_conteo
0,2,True,False
1,3,True,False
2,4,True,False


### SHAP por horizonte de F+A3 (semilla 17)

En S4_s3 el margen de persistencia entra como `base_margin`, así que SHAP describe la desviación respecto de la
persistencia, no el nivel completo.

In [16]:
_, modelos = sp.predecir_e3(matrices, 17, CONF["formulacion"], devolver_modelos=True, columnas=COL_FA3, origenes=o_a3,
                           hiperparametros=CONF["hiperparametros"], dispersion=CONF["dispersion"])
shap = {b: sp.importancia_shap_por_horizonte(modelos[b]["regresor"], modelos[b]["x_reg"])
        for b in [*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE]}
ruta_shap = d.guardar_figura(ev.figura_shap(shap, "SHAP por horizonte, F+A3 (semilla 17)"), FIGURAS_FE_HIPERPARAMETROS,
                             "shap_por_horizonte_F_A3")
pd.concat(shap, names=["bloque", "horizonte"]).round(4).T

bloque                              temporada_2022                  \
horizonte                                      h=2     h=3     h=4   
hist_lag_a                                  0.0605  0.0841  0.1086   
hist_lag_b                                  0.1722  0.1462  0.1230   
hist_media_4                                0.2617  0.2116  0.1601   
hist_semanas_pos_4                          0.0810  0.0736  0.0657   
semana_epi_seno                             0.3227  0.3940  0.4410   
semana_epi_coseno                           0.0355  0.0988  0.1696   
propia_semanas_desde_ultimo_caso            0.8063  0.7266  0.6483   
propia_pendiente_4                          0.0476  0.0471  0.0478   
propia_log_casos_temporada_anterior         0.3116  0.3327  0.3448   
hum_rel_media_media5_h2                     0.0400  0.0623  0.0784   
precip_total_mm_media5_h2                   0.2775  0.3221  0.3543   

bloque                              temporada_2023                  \
horizonte                                      h=2     h=3     h=4   
hist_lag_a                                  0.0563  0.0828  0.1018   
hist_lag_b                                  0.0399  0.0277  0.0174   
hist_media_4                                0.3506  0.3050  0.2557   
hist_semanas_pos_4                          0.0582  0.0564  0.0530   
semana_epi_seno                             0.2595  0.3052  0.3296   
semana_epi_coseno                           0.1011  0.2027  0.3052   
propia_semanas_desde_ultimo_caso            0.9055  0.8056  0.7304   
propia_pendiente_4                          0.0352  0.0334  0.0326   
propia_log_casos_temporada_anterior         0.1757  0.1997  0.2120   
hum_rel_media_media5_h2                     0.0628  0.0848  0.1018   
precip_total_mm_media5_h2                   0.1784  0.2061  0.2289   

bloque                              temporada_2024                  
horizonte                                      h=2     h=3     h=4  
hist_lag_a                                  0.0514  0.0884  0.1185  
hist_lag_b                                  0.0334  0.0225  0.0406  
hist_media_4                                0.2078  0.1663  0.1312  
hist_semanas_pos_4                          0.0656  0.0630  0.0593  
semana_epi_seno                             0.2337  0.2801  0.3184  
semana_epi_coseno                           0.0443  0.0940  0.1688  
propia_semanas_desde_ultimo_caso            0.7751  0.7086  0.6477  
propia_pendiente_4                          0.0143  0.0174  0.0171  
propia_log_casos_temporada_anterior         0.1935  0.2160  0.2354  
hum_rel_media_media5_h2                     0.1032  0.1367  0.1554  
precip_total_mm_media5_h2                   0.1801  0.2186  0.2461

## Guardado y registro

In [17]:
for nombre, r in (("F_A3", fa3), ("F_interseccion_A3", f_int), ("F_completa", f_completa)):
    pd.concat(r["predicciones"].values(), ignore_index=True).to_csv(DESTINO / f"predicciones_{nombre}.csv.gz", index=False)
ruta = d.guardar_seccion_metricas(METRICAS_FE_HIPERPARAMETROS, "complemento_clima", {
    "preregistro": cc.PREREGISTRO_COMPLEMENTO, "filas_excluidas_A3": excluidas,
    "conteo": d.a_json(conteo), "condicion_conteo": d.a_json(cond_conteo),
    "veredicto_alerta": d.a_json(veredicto), "decision": decision,
    "metricas": d.a_json(lectura), "antelacion": d.a_json(antelacion),
    "antelacion_frente_a_persistencia_semilla_17": d.a_json(persistencia),
    "recall_por_tipo_semana": d.a_json(recall), "conteo_por_fase": d.a_json(fases),
    "cumplimiento_hu0007_4": d.a_json(hu), "diagnostico_2022_h4": d.a_json(diag_2022_h4),
    "robustez_solo_lectura": {"conteo": d.a_json(conteo_rob), "signos": d.a_json(signos),
                              "condicion_conteo": d.a_json(cc.condicion_conteo(conteo_rob))},
    "shap_por_horizonte_semilla_17": {b: d.a_json(t.reset_index(names="horizonte")) for b, t in shap.items()},
    "comparaciones_acumuladas": pr.comparaciones()["total_encargo"],
})
print(ruta.relative_to(ROOT))

docs\modeling\metricas\fe_hiperparametros.json


In [18]:
reporte = fh.reporte_mlflow("fe_hiperparametros_complemento_clima",
                            {"F+A3": fa3["resumen"]["medianas"], "F∩A3": f_int["resumen"]["medianas"]},
                            hiperparametros=CONF["hiperparametros"])
reporte["formulacion"] = CONF["formulacion"]
reporte["decision"] = decision
registrar_o_avisar("fe_hiperparametros_complemento_clima", reporte, metricas_path=METRICAS_FE_HIPERPARAMETROS,
                   salida_dir=DESTINO, gold_dir=None, huella=fh.huella_directorio(DESTINO),
                   descripcion=f"Complemento de clima (A3) sobre la configuración final: adoptada {decision['adoptada']}")

MLflow: ya existe el run 6afc4549fd214a1497636319b3e6844c con huella e3859d58c464 en 'denguekay/fe_hiperparametros_complemento_clima'; no se registra otro


'6afc4549fd214a1497636319b3e6844c'